# Exploratory Analysis — AI Ad Review Quality Framework

This notebook provides an initial exploration of the synthetic ad review dataset.
It covers:
- Dataset overview and distributions
- LLM and human accuracy by policy category
- Confidence score distribution
- Market and language disagreement patterns
- BPO team and reviewer tenure analysis
- Appeal reversal patterns

**Prerequisites**: Run the pipeline before opening this notebook.
```
python src/generate_dataset.py
python src/prepare_data.py
python src/evaluate_quality.py
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

DATA_PATH = Path('../data/processed/ad_reviews_enriched.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['created_date'])

bool_cols = [
    'is_llm_correct', 'is_human_correct', 'is_human_ai_agreement',
    'is_advertiser_over_rejection', 'is_policy_risk_miss',
    'is_low_confidence_case', 'is_high_risk_approval_miss',
    'is_policy_ambiguous', 'is_appeal_reversed', 'appeal_submitted',
]
for c in bool_cols:
    if c in df.columns:
        df[c] = df[c].astype(bool)

print(f'Dataset loaded: {len(df):,} records, {len(df.columns)} columns')
df.head(3)

## 1. Dataset Overview

In [ ]:
print('=== Overall Quality KPIs ===')
total = len(df)
appealed = df['appeal_submitted'].sum()
reversed_ = df['is_appeal_reversed'].sum()

kpis = {
    'Total cases': f'{total:,}',
    'LLM accuracy': f"{df['is_llm_correct'].mean():.1%}",
    'Human accuracy': f"{df['is_human_correct'].mean():.1%}",
    'Human-AI agreement': f"{df['is_human_ai_agreement'].mean():.1%}",
    'Over-rejection rate': f"{df['is_advertiser_over_rejection'].mean():.1%}",
    'Policy risk miss rate': f"{df['is_policy_risk_miss'].mean():.1%}",
    'High-risk approval misses': f"{df['is_high_risk_approval_miss'].sum():,}",
    'Low-confidence case rate': f"{df['is_low_confidence_case'].mean():.1%}",
    'Appeal submission rate': f"{appealed / total:.1%}",
    'Appeal reversal rate': f"{reversed_ / appealed:.1%}" if appealed > 0 else 'N/A',
}
for k, v in kpis.items():
    print(f'  {k:<30} {v}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Policy category distribution
cat_counts = df['policy_category'].value_counts()
axes[0].barh(cat_counts.index, cat_counts.values, color='#4C78A8')
axes[0].set_title('Cases by Policy Category')
axes[0].set_xlabel('Count')

# Market distribution
mkt_counts = df['market'].value_counts()
axes[1].bar(mkt_counts.index, mkt_counts.values, color='#54A24B')
axes[1].set_title('Cases by Market')
axes[1].set_ylabel('Count')

# Risk level distribution
risk_counts = df['risk_level'].value_counts().reindex(['low', 'medium', 'high'])
axes[2].bar(risk_counts.index, risk_counts.values, color=['#54A24B', '#F58518', '#E45756'])
axes[2].set_title('Cases by Risk Level')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 2. LLM and Human Accuracy by Policy Category

In [ ]:
cat_acc = (
    df.groupby('policy_category')
    .agg(
        llm_accuracy=('is_llm_correct', 'mean'),
        human_accuracy=('is_human_correct', 'mean'),
        agreement=('is_human_ai_agreement', 'mean'),
        cases=('ad_id', 'count'),
    )
    .sort_values('llm_accuracy')
)

fig, ax = plt.subplots(figsize=(10, 6))
y = range(len(cat_acc))
ax.barh([i + 0.2 for i in y], cat_acc['human_accuracy'], height=0.35,
        label='Human accuracy', color='#54A24B', alpha=0.85)
ax.barh([i - 0.2 for i in y], cat_acc['llm_accuracy'], height=0.35,
        label='LLM accuracy', color='#4C78A8', alpha=0.85)
ax.set_yticks(list(y))
ax.set_yticklabels(cat_acc.index)
ax.set_xlabel('Accuracy')
ax.set_title('LLM vs Human Accuracy by Policy Category', fontsize=12, fontweight='bold')
ax.axvline(0.8, color='gray', linestyle='--', alpha=0.5, label='80% threshold')
ax.legend()
plt.tight_layout()
plt.show()

print(cat_acc.to_string())

## 3. LLM Confidence Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall distribution
axes[0].hist(df['llm_confidence_score'], bins=30, color='#72B7B2', edgecolor='white')
axes[0].axvline(0.60, color='#E45756', linestyle='--', label='Low conf. threshold (0.60)')
axes[0].set_title('LLM Confidence Score Distribution')
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Count')
axes[0].legend()

# Error rate by confidence bucket
df['conf_bucket'] = pd.cut(
    df['llm_confidence_score'],
    bins=[0, 0.50, 0.60, 0.70, 0.80, 1.01],
    labels=['<0.50', '0.50–0.60', '0.60–0.70', '0.70–0.80', '>0.80']
)
err_by_conf = df.groupby('conf_bucket', observed=True)['is_llm_correct'].agg(lambda x: 1 - x.mean())
axes[1].bar(err_by_conf.index, err_by_conf.values,
            color=['#E45756', '#E45756', '#F58518', '#F58518', '#54A24B'])
axes[1].set_title('LLM Error Rate by Confidence Bucket')
axes[1].set_xlabel('Confidence Bucket')
axes[1].set_ylabel('Error Rate')
for i, v in enumerate(err_by_conf.values):
    axes[1].text(i, v + 0.003, f'{v:.1%}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 4. Over-Rejection vs Policy Risk Miss by Category

In [ ]:
err_profile = (
    df.groupby('policy_category')
    .agg(
        over_rejection=('is_advertiser_over_rejection', 'mean'),
        risk_miss=('is_policy_risk_miss', 'mean'),
    )
    .sort_values('over_rejection', ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(err_profile))
ax.bar(x, err_profile['over_rejection'], label='Over-rejection', color='#E45756', alpha=0.8)
ax.bar(x, [-v for v in err_profile['risk_miss']], label='Risk miss', color='#F58518', alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(err_profile.index, rotation=40, ha='right')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Rate')
ax.set_title('Over-Rejection (above) vs Policy Risk Miss (below) by Category',
             fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Market and Language Disagreement

In [ ]:
market_dis = (
    df.groupby('market')
    .agg(
        llm_accuracy=('is_llm_correct', 'mean'),
        disagreement=('is_human_ai_agreement', lambda x: 1 - x.mean()),
        avg_review_time=('review_time_seconds', 'mean'),
    )
    .sort_values('disagreement', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(market_dis.index, market_dis['disagreement'], color='#4C78A8')
axes[0].set_title('Human-AI Disagreement Rate by Market')
axes[0].set_ylabel('Disagreement Rate')
for i, v in enumerate(market_dis['disagreement']):
    axes[0].text(i, v + 0.003, f'{v:.1%}', ha='center', fontsize=9)

axes[1].bar(market_dis.index, market_dis['avg_review_time'], color='#72B7B2')
axes[1].set_title('Average Review Time by Market (seconds)')
axes[1].set_ylabel('Seconds')
for i, v in enumerate(market_dis['avg_review_time']):
    axes[1].text(i, v + 1, f'{v:.0f}s', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(market_dis.to_string())

## 6. BPO Team and Reviewer Tenure Analysis

In [ ]:
bpo_acc = (
    df.groupby('bpo_team')['is_human_correct']
    .mean()
    .sort_values()
)
tenure_err = (
    df.groupby('reviewer_tenure')['is_human_correct']
    .agg(lambda x: 1 - x.mean())
    .reindex(['new', 'intermediate', 'experienced'])
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(bpo_acc.index, bpo_acc.values, color='#54A24B')
axes[0].set_title('Human Accuracy by BPO Team')
axes[0].set_xlabel('Human Accuracy')
for i, v in enumerate(bpo_acc.values):
    axes[0].text(v + 0.002, i, f'{v:.1%}', va='center', fontsize=9)

colors = ['#E45756', '#F58518', '#54A24B']
axes[1].bar(tenure_err.index, tenure_err.values, color=colors)
axes[1].set_title('Human Error Rate by Reviewer Tenure')
axes[1].set_ylabel('Error Rate')
for i, v in enumerate(tenure_err.values):
    axes[1].text(i, v + 0.003, f'{v:.1%}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Appeal Reversal Analysis

In [ ]:
appealed_df = df[df['appeal_submitted']].copy()

rev_by_cat = (
    appealed_df.groupby('policy_category')['is_appeal_reversed']
    .mean()
    .sort_values(ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(rev_by_cat.index, rev_by_cat.values, color='#F58518')
axes[0].set_title('Appeal Reversal Rate by Policy Category')
axes[0].set_xlabel('Reversal Rate')
for i, v in enumerate(rev_by_cat.values):
    axes[0].text(v + 0.005, i, f'{v:.1%}', va='center', fontsize=8)

rev_by_label = (
    appealed_df.groupby('llm_label')['is_appeal_reversed']
    .mean()
    .sort_values(ascending=False)
)
axes[1].bar(rev_by_label.index, rev_by_label.values, color='#E45756')
axes[1].set_title('Appeal Reversal Rate by LLM Label')
axes[1].set_ylabel('Reversal Rate')
for i, v in enumerate(rev_by_label.values):
    axes[1].text(i, v + 0.005, f'{v:.1%}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f'Total appeals: {len(appealed_df):,}')
print(f'Total reversals: {appealed_df["is_appeal_reversed"].sum():,}')
print(f'Overall reversal rate: {appealed_df["is_appeal_reversed"].mean():.1%}')

## 8. High-Risk Approval Misses

In [ ]:
hr_misses = df[df['is_high_risk_approval_miss']].copy()
print(f'Total high-risk approval misses: {len(hr_misses):,}')
print()

print('By policy category:')
print(hr_misses['policy_category'].value_counts().to_string())
print()

print('By market:')
print(hr_misses['market'].value_counts().to_string())
print()

print('LLM confidence for high-risk misses:')
print(hr_misses['llm_confidence_score'].describe().round(3).to_string())

## Summary

This exploratory analysis confirms the expected patterns in the synthetic dataset:

- **LLM accuracy varies significantly by policy category** — lowest for Landing Page Issue and Financial Product Claim, highest for Safe Ad and Scam cases.
- **Low-confidence LLM decisions have higher error rates** — the 0.60 threshold is a reliable routing signal.
- **Non-English markets show higher disagreement** — local policy nuance is an important calibration gap.
- **New reviewers diverge most from golden labels** — calibration training is most needed for this group.
- **Appeal reversals are concentrated in over-rejected categories** — Safe Ad and Misleading Claim show the highest reversal rates.
- **High-risk approval misses are limited but real** — concentrated in regulated categories.

See `docs/recommendations.md` for actionable next steps.